# 06_dashboard_export

**Input:** `rides_features.parquest` (built by notebook 03)

**Output:** five small summary files in `dashboard/data/` that Power BI dashboard reads

**Why this exists:** So anyone can rebuild the dashboard's inputs by running this notebook. The repo only needs to include the five summary files, not the ride-level data.


In [9]:
import pandas as pd
from pathlib import Path

OUT = Path("../dashboard/data")
OUT.mkdir(parents=True, exist_ok=True)

rides = pd.read_parquet("../data/clean/rides_features.parquet",
    columns=["member_casual", "rideable_type", "day_of_week", "day_type",
             "hour", "season", "ride_min", "start_station_name"])

def tidy(df, key):
    t = df.groupby([key, "member_casual"], observed=True).size().rename("rides").reset_index()
    t["share_pct"] = (t["rides"] / t.groupby("member_casual", observed=True)["rides"].transform("sum") * 100).round(1)
    return t

#by_day.csv: ride counts and each group's share of its own rides, by day of week (day_num keeps days in order)

by_day = tidy(rides, "day_of_week")
by_day["day_num"] = by_day["day_of_week"].astype(str).map(
    {"Monday": 1, "Tuesday": 2, "Wednesday": 3, "Thursday": 4, "Friday": 5, "Saturday": 6, "Sunday": 7})
by_day.to_csv(OUT / "by_day.csv", index=False)

#by_weekday_hour.csv: same measures, by hour, weekdays only

tidy(rides[rides["day_type"] == "Weekday"], "hour").to_csv(OUT / "by_weekday_hour.csv", index=False)

#by_season.csv: same measure, by season (season_num keeps season in order)

by_season = tidy(rides, "season")
by_season["season_num"] = by_season["season"].map({"Winter": 1, "Spring": 2, "Summer": 3, "Fall": 4})
by_season.to_csv(OUT / "by_season.csv", index=False)

#by_bike_type.csv: median ride length in minutes and ride count, for each bike type and rider group

(rides.groupby(["rideable_type", "member_casual"], observed=True)["ride_min"]
      .agg(median_min="median", rides="size").round(1).reset_index()
      .to_csv(OUT / "by_bike_type.csv", index=False))

#by_station.csv: one row per start station with 3,000+ casual rides: casual share of all rides there and share of its casual rides that start in weekday commute hours

st = rides.dropna(subset=["start_station_name"]).copy()
st["commute"] = (st["day_type"] == "Weekday") & st["hour"].isin([7, 8, 16, 17])
cas = st[st["member_casual"] == "casual"]
mem_base = st.loc[st["member_casual"] == "member", "commute"].mean() * 100
cas_base = cas["commute"].mean() * 100

by = cas.groupby("start_station_name", observed=True).agg(
    casual_rides=("commute", "size"), commute_share=("commute", "mean"))
tot = st.groupby("start_station_name", observed=True).size()
by["casual_share"] = by["casual_rides"] / tot.reindex(by.index) * 100
by["commute_share"] *= 100
big = by[by["casual_rides"] >= 3000].copy()

#group: Attraction = casual share of 50% or more and commute share below the casual average; member-like = commute share at or above the member average; all others = other

big["group"] = "Other"
big.loc[(big["commute_share"] < cas_base) & (big["casual_share"] >= 50), "group"] = "Attraction"
big.loc[big["commute_share"] >= mem_base, "group"] = "Member-like"
big.round(1).reset_index().rename(columns={"commute_share": "commute"}).to_csv(OUT / "by_station.csv", index=False)